# 02 · Data Cleaning

**Project:** Data Analyst – Mental Health (Canada) · **Pipeline step:** 2 of 10

> **Skeleton only.** This notebook currently just wires up the libraries and the
> `data/raw` → `data/processed` connections so the team can start cleaning tomorrow.
> The cleaning logic goes under section 4.


## 1 · Libraries

In [1]:
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("pandas", pd.__version__, "| numpy", np.__version__)


pandas 3.0.5 | numpy 2.5.2


## 2 · Paths (`data/raw` → `data/processed`)

In [2]:
def find_root(start: Path) -> Path:
    """Walk up until we find the folder that contains data/raw (works from repo root or /notebooks)."""
    for p in [start, *start.parents]:
        if (p / "data" / "raw").is_dir():
            return p
    raise FileNotFoundError("Could not find data/raw above " + str(start))

ROOT = find_root(Path.cwd())
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

print("root     :", ROOT)
print("raw      :", RAW)
print("processed:", PROCESSED)


root     : /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-
raw      : /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/raw
processed: /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/processed


## 3 · Load the raw datasets

Same registry as `01_data_understanding.ipynb`. StatCan CSVs need `utf-8-sig` (BOM).
The CIHI Excel workbook is loaded separately (only the two hidden data sheets are usable).

In [3]:
DATASETS = {
    "perceived_mh_annual":        {"file": "StatCan 13-10-0972 – perceived mental health.csv",                                              "kind": "statcan_long"},
    "suicidal_thoughts":          {"file": "Catalogue Entry Mental health characteristics and suicidal thoughts.csv",                        "kind": "statcan_long"},
    "stress_coping":              {"file": "Catalogue Entry Mental health characteristics Ability to handle stress and sources of stress.csv","kind": "statcan_long"},
    "perceived_health_quarterly": {"file": "Catalogue Entry Mental health indicators.csv",                                                   "kind": "statcan_long"},
    "cchs_mh_disorders":          {"file": "Catalogue Entry Perceived health, by gender and province.csv",                                   "kind": "statcan_long"},
    "cihi_mh_services":           {"file": "health services for mental illness and alcoholdrug induced disorders.csv",                       "kind": "cihi_vizconfig"},
    "cihi_children_youth":        {"file": "care-children-youth-with-mental-disorders-data-tables-en.xlsx",                                  "kind": "excel_multitable"},
    "mhacs_2022_pumf":            {"file": "MHACS 2022 Public Use Microdata.csv",                                                            "kind": "microdata"},
}

def load_dataset(key: str) -> pd.DataFrame:
    spec = DATASETS[key]
    path = RAW / spec["file"]
    if spec["kind"] in ("statcan_long", "cihi_vizconfig"):
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    if spec["kind"] == "microdata":
        return pd.read_csv(path, low_memory=False)
    raise ValueError(f"{key}: kind={spec['kind']} is loaded separately (see below)")

# CSV datasets -> raw[...]
raw = {k: load_dataset(k) for k, v in DATASETS.items() if v["kind"] != "excel_multitable"}

# CIHI workbook: the two machine-readable sheets (title row 0, headers row 1)
_xls = pd.ExcelFile(RAW / DATASETS["cihi_children_youth"]["file"])
raw_excel = {}
for _sheet in [s for s in _xls.sheet_names if s.endswith("_to hide")]:
    _t = _xls.parse(_sheet, header=None)
    _body = _t.iloc[2:].reset_index(drop=True)
    _body.columns = [str(h).replace("\n", " ").strip() for h in _t.iloc[1]]
    raw_excel[_sheet] = _body

for k, df in raw.items():
    print(f"{k:28} {df.shape}")
for k, df in raw_excel.items():
    print(f"{k:28} {df.shape}  (excel)")


perceived_mh_annual          (936, 18)
suicidal_thoughts            (8208, 18)
stress_coping                (27360, 18)
perceived_health_quarterly   (6318, 17)
cchs_mh_disorders            (160992, 18)
cihi_mh_services             (264, 14)
mhacs_2022_pumf              (9861, 602)
Table8DATA_to hide           (216, 13)  (excel)
Table13DATA_to hide          (216, 13)  (excel)


## 4 · Cleaning — TO BE COMPLETED BY THE TEAM

Write cleaned outputs to `PROCESSED / "..."`. Suggested tasks (see `docs/data_dictionary.md` §A):

- [ ] StatCan tables: melt to tidy long; pivot `Characteristics`/`Statistics` into `value` / `ci_low` / `ci_high` / `cv`
- [ ] Standardise `GEO` to one canonical province list; handle region rollups separately
- [ ] Parse `REF_DATE` (year / year-range / year-month) into a real date/period
- [ ] Apply `SCALAR_FACTOR` (×1000 where `thousands`); keep `STATUS` as `quality_flag`; **do not impute** suppressed values
- [ ] Filter to the agreed analysis window
- [ ] `cihi_mh_services`: unpivot chart-config rows → `indicator | breakdown | group | value | ci_low | ci_high`
- [ ] `cihi_children_youth`: split each `95% CI` string into `ci_low` / `ci_high`; tidy long
- [ ] `mhacs_2022_pumf`: replace non-response codes (6/7/8/9, 96, 996, 99.6 …) with NaN for the selected variables only
- [ ] Save each cleaned dataset to `data/processed/` and note row counts


In [4]:
# team: add cleaning code here
